# HAVEN — Agent Pipeline Runner
**Notebook:** 01_pipeline.ipynb  
**Purpose:** Run Agent 1 → Agent 2 → Agent 3 on all families in families.json and save outputs to `outputs/pipeline/`  
**Models:** Claude Haiku 4.5 (Agent 1), Claude Sonnet 4.6 (Agent 2 + 3)  
**Prompts used:** Agent 1 v7, Agent 2 v9, Agent 3 v1 (see evals/prompts/prompt_log.md)

---
Run cells in order. Outputs are saved as JSON after each agent stage so you can inspect intermediate results without re-running everything.

## 1. Setup

In [1]:
# Install dependencies if needed
# !pip install anthropic openai

import anthropic
import json
import os
import time
from pathlib import Path

# Paths
BASE_DIR = Path("../")
DATA_DIR = BASE_DIR / "data"
OUTPUT_DIR = Path("outputs/pipeline")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# API client
client = anthropic.Anthropic(api_key=os.environ.get("ANTHROPIC_API_KEY"))

print("Setup complete.")
print(f"Data directory: {DATA_DIR.resolve()}")
print(f"Output directory: {OUTPUT_DIR.resolve()}")

Setup complete.
Data directory: /Users/samvedasharma/Desktop/Documents/AI Product Manager Portfolio/haven/data
Output directory: /Users/samvedasharma/Desktop/Documents/AI Product Manager Portfolio/haven/evals/outputs/pipeline


## 2. Load Data

In [2]:
with open(DATA_DIR / "families.json") as f:
    families = json.load(f)

with open(DATA_DIR / "helpers.json") as f:
    helpers = json.load(f)

with open(DATA_DIR / "golden_set.json") as f:
    golden_set = json.load(f)

helpers_json_str = json.dumps(helpers, indent=2)

print(f"Families loaded: {len(families)}")
print(f"Helpers loaded: {len(helpers)}")
print(f"Golden set entries: {len(golden_set)}")

Families loaded: 17
Helpers loaded: 20
Golden set entries: 17


## 3. Agent 1 — Care Need Articulator
**Model:** Claude Haiku 4.5  
**Prompt version:** v4 (locked)  
**Job:** Convert family free-text description → structured care profile JSON

In [3]:
AGENT_1_PROMPT = """You are an expert care coordinator helping families find the right care helper.

Extract a structured care profile from the family description provided.

Return ONLY a JSON object with these exact fields:
{
  "condition": "the primary condition or care need",
  "care_type": "type of care required — infer from the description if not explicitly stated",
  "schedule": "when care is needed — include specific hours if mentioned",
  "budget": "hourly budget range if stated (e.g. '$18-$22/hr') — null if not mentioned",
  "household": "who lives in the home",
  "must_haves": ["personality and behavioral non-negotiables"],
  "deal_breakers": [
    {"text": "the requirement the helper must meet, phrased affirmatively", "type": "hard or soft"}
  ],
  "required_skills": ["practical abilities the job needs — e.g. cooking, light housekeeping, pet care, mobility assistance — whether stated or clearly implied"],
  "explicit_qualifications": ["formal certifications or training the family directly asked for — empty array if none mentioned"],
  "inferred_qualifications": ["formal certifications or training implied by a clearly stated condition — e.g. dementia care certification, first aid training. Do NOT include general skills or infer clinical certifications from vague descriptions"],
  "clarification_needed": [
    {
      "field": "schedule or budget only",
      "reason": "why this field is missing or too vague to match on",
      "question": "the specific follow-up question to ask the family"
    }
  ],
  "confidence": 0.0
}

Rules:
- Set confidence between 0 and 1. If schedule or budget is missing, set below 0.75.
- care_type: always infer from the description — never leave blank.
- budget: extract as a range if stated (e.g. "$18–$22/hr"). If not mentioned, set to null and flag in clarification_needed.
- required_skills: practical abilities only — cooking, cleaning, pet care, transfers etc. Not certifications.
- inferred_qualifications: only infer formal certifications when the condition is clearly and specifically named (e.g. "dementia", "Parkinson's", "MS"). If the description is vague or mild (e.g. "sometimes forgets things", "aging in place", "memory concerns"), do not infer clinical certifications — leave inferred_qualifications empty.
- clarification_needed: ONLY flag schedule (if vague or missing) or budget (if not stated). Do not flag any other fields.
- deal_breakers: capture all disqualifying conditions — including cultural, demographic, language, and gender requirements. Classify each as:
  - "hard": family used explicit mandatory language ("required", "must", "need", "mandatory", "non-negotiable") OR it is a safety baseline (abuse history, violent behaviour, criminal record) OR the family describes past negative experience where helpers were dismissed or did not work out specifically because of this requirement — past demonstrated behavior counts as mandatory
  - "soft": family used preference language ("preferred", "good to have", "ideally", "would like") OR you inferred it from the condition or context without the family stating it explicitly
- Phrase all deal-breaker text affirmatively — write what the helper MUST BE or MUST HAVE, not what they must not be. Write "Helper must be a non-smoker", not "Helper should not be a smoker". Write "Helper must have South Asian cultural background", not "Helper is not from South Asian cultural background". Write "Helper must be female", not "Helper must not be male".
- When care_type is companion care AND the family explicitly signals they want a non-clinical, low-key, or unfussy presence (e.g. "dislikes fuss", "calm low-key", "does not want medical feel"), add a soft deal-breaker: "Helper must have a calm, non-clinical companion care manner\""""


def run_agent_1(family_description: str, family_id: str) -> dict:
    """Run Agent 1 on a single family description. Returns parsed care profile."""
    message = client.messages.create(
        model="claude-haiku-4-5",
        max_tokens=2048,
        messages=[
            {
                "role": "user",
                "content": f"{AGENT_1_PROMPT}\n\nFamily description:\n\"{family_description}\""
            }
        ]
    )
    raw = message.content[0].text.strip()
    if raw.startswith("```"):
        raw = raw.split("```")[1]
        if raw.startswith("json"):
            raw = raw[4:]
    try:
        parsed = json.loads(raw)
    except json.JSONDecodeError:
        parsed = {"error": "failed to parse JSON", "raw": raw}
    return {"family_id": family_id, "care_profile": parsed}


print("Agent 1 function defined.")

Agent 1 function defined.


In [4]:
# Run Agent 1 on all families
agent_1_outputs = []

for family in families:
    family_id = family["family_id"]
    description = family["description"]
    print(f"Running Agent 1 on {family_id}...", end=" ")
    result = run_agent_1(description, family_id)
    agent_1_outputs.append(result)
    print(f"confidence: {result['care_profile'].get('confidence', 'N/A')}")
    time.sleep(0.5)  # avoid rate limiting

# Save outputs
with open(OUTPUT_DIR / "agent_1_outputs.json", "w") as f:
    json.dump(agent_1_outputs, f, indent=2)

print(f"\nAgent 1 complete. Outputs saved to {OUTPUT_DIR / 'agent_1_outputs.json'}")

Running Agent 1 on F001... confidence: 0.68
Running Agent 1 on F002... confidence: 0.95
Running Agent 1 on F003... confidence: 0.95
Running Agent 1 on F004... confidence: 0.92
Running Agent 1 on F005... confidence: 0.95
Running Agent 1 on F006... confidence: 0.62
Running Agent 1 on F007... confidence: 0.68
Running Agent 1 on F008... confidence: 0.68
Running Agent 1 on F009... confidence: 0.68
Running Agent 1 on F010... confidence: 0.82
Running Agent 1 on F011... confidence: 0.95
Running Agent 1 on F012... confidence: 0.95
Running Agent 1 on F013... confidence: 0.95
Running Agent 1 on F014... confidence: 0.92
Running Agent 1 on F015... confidence: 0.72
Running Agent 1 on F016... confidence: 0.95
Running Agent 1 on F017... confidence: 0.82

Agent 1 complete. Outputs saved to outputs/pipeline/agent_1_outputs.json


In [5]:
# Spot check — inspect one care profile
INSPECT_FAMILY = "F017"  # change this to inspect a different family

result = next(r for r in agent_1_outputs if r["family_id"] == INSPECT_FAMILY)
print(json.dumps(result, indent=2))

{
  "family_id": "F017",
  "care_profile": {
    "condition": "Mild cognitive impairment with aging in place",
    "care_type": "Companion care with light housekeeping and ADL support",
    "schedule": "Weekday mornings \u2014 specific hours not specified",
    "budget": "$25\u2013$30/hr",
    "household": "83-year-old grandmother (lives with family, implied)",
    "must_haves": [
      "Reliable and consistent",
      "Patient and compassionate",
      "Good communicator",
      "Trustworthy with medication reminders"
    ],
    "deal_breakers": [
      {
        "text": "Helper must be from a South Asian cultural background",
        "type": "hard"
      },
      {
        "text": "Helper must have a calm, non-clinical companion care manner",
        "type": "soft"
      }
    ],
    "required_skills": [
      "Meal preparation",
      "Light housekeeping",
      "Medication reminders",
      "Companionship and conversation",
      "Dementia-friendly communication"
    ],
    "explic

## 4. Agent 2 — Helper Profile Scorer
**Model:** Claude Sonnet 4.6  
**Prompt version:** v9 (active)  
**Job:** Score helper pool against care profile → return top 3 ranked helpers with scores

In [6]:
AGENT_2_PROMPT = """You are a care matching specialist. Your job is to score a pool of care helpers against a family's care profile and return the top 3 best-fit helpers.

You will be given:
1. A care profile JSON (the family's needs)
2. A list of helper profiles JSON

Apply the following decision logic internally — do not write out steps, per-helper scoring, or intermediate reasoning. Output ONLY the final JSON.

STEP 1 — BUDGET GATE
Only apply if care_profile.budget is not null.
  - Parse the upper bound of the family's budget range
  - Calculate the exclusion threshold: upper_bound × 1.20
  - Silently exclude any helper whose hourly_rate exceeds this threshold
  - If care_profile.budget is null, skip this step entirely
  Example: family budget is $13–$15/hr → upper bound = $15 → threshold = $18.00. A helper at $16/hr passes (16 < 18). A helper at $17/hr passes (17 < 18). A helper at $19/hr is excluded (19 > 18). Apply this threshold consistently — do not exclude helpers who are under the threshold.

STEP 2 — DEAL-BREAKER CHECK
For each helper not excluded in Step 1, check each deal-breaker against what is known or inferable about the helper.
Each deal-breaker in the care profile has a type field: "hard" or "soft". The deal-breaker describes helper condition that causes disqualification.
If any single deal-breaker is exhibited by the helper or is strongly inferable from their profile, exclude that helper entirely. Example, if the deal breaker is "smoker", exclude all helpers who smoke.
Log each excluded helper with the specific deal-breaker that caused exclusion and its type.

Only exclude a helper if a deal-breaker is clearly confirmed or strongly inferable. Do not exclude on weak signals.

STEP 2.5 — SCHEDULE VIABILITY CHECK
Run this after Step 2.

Check whether the family's required schedule is non-standard — meaning any window that falls between 6pm and 9am, or is weekend-only.

If yes: count how many helpers surviving Step 2 have any confirmed or inferable availability overlap with those required hours.

If zero surviving helpers cover the required hours — this is a no-match scenario. Do the following:
  1. Restore any helpers excluded in Step 2 solely because of "soft" deal-breakers. Keep all "hard" deal-breaker exclusions intact.
  2. Proceed to Step 3 with this restored pool.
  3. In Step 3, apply the following schedule_fit override for the no-match pool:
     - Helpers who list "Flexible" days or hours: score schedule_fit = 2.0 (they can potentially cover non-standard hours)
     - Helpers with confirmed daytime-only schedules (e.g. 8am–1pm, 9am–2pm): score schedule_fit = 0.0–0.5 (daytime schedule has no overlap with the overnight/non-standard window)
     Flexible schedule must score higher than confirmed daytime-only for non-standard families.
  4. After Step 3 scoring, override the composite ranking: place all restored helpers above any helper whose schedule_fit ≤ 0.5, regardless of composite score. Among restored helpers, rank by composite. Among schedule_fit ≤ 0.5 helpers, rank by composite.
  5. Set "no_match": true in the output JSON. In step_reasoning.schedule_viability, record which soft deal-breakers were suspended and which helpers were restored.

If the required schedule is standard (daytime/weekday) OR at least one surviving helper has schedule overlap — skip this step entirely and proceed to Step 3 normally. Set "no_match": false.

STEP 3 — SCORE EACH REMAINING HELPER
Score each helper across 4 dimensions. Use one decimal point only.
If data for a dimension is missing and cannot be inferred, score that dimension as 0 and note it in profile_gaps.

Schedule Fit (0–3.0):
  - 3.0: helper's days and hours fully cover the family's need
  - 2.0–2.9: partial overlap — minor gaps in days or hours (e.g. missing one day, or start/end time off by 30 minutes)
  - 1.0–1.9: limited overlap — helper covers roughly half to three-quarters of the required hours window, or is missing multiple required days
  - 0.5: helper covers less than half the required hours window
  - 0.0: no overlap, or schedule completely unspecified and cannot be inferred
  Coverage rule: score based on the proportion of the family's total required hours window the helper actually covers — a helper who covers 4 of 8 required hours is in the limited overlap band, not partial overlap. Calculate overlap accounting for both the helper's start time and end time against the family's required window — neither end is discounted.

Care Experience (0–3.0):
  - 2.5–3.0: direct, substantial experience with the exact condition stated. Do not award this band if the care profile condition is vague or mild (e.g. "sometimes forgets things", "aging in place", "memory concerns") and the helper's experience is with a more severe or clinical version of a related condition
  - 1.5–2.4: some related experience, limited direct condition experience, or strong general elderly care for a mild/vague condition
  - 0.5–1.4: general elderly care only, no specific condition match
  - 0.0: no relevant experience evident or cannot be assessed

Skills Match (0–2.0):
  - Score proportionally based on how many required_skills from the care profile the helper demonstrably covers
  - Full coverage = 2.0, each clearly missing skill reduces proportionally
  - 0.0 if skills field is empty and cannot be inferred from bio

Qualifications Match (0–2.0):
Score ONLY against the care profile's explicit_qualifications and inferred_qualifications fields.
Do not reward certifications that do not appear in those fields, regardless of how many the helper holds.
  - 1.8–2.0: holds one or more certifications that exactly match explicit_qualifications or inferred_qualifications
  - 1.0–1.7: holds a certification with partial overlap — same domain as a required qualification, but not an exact match
  - 0.0: no certifications matching explicit or inferred qualifications — score 0 regardless of total certifications held

Volume rule: 5 relevant certifications scores no higher than 1 relevant certification. Only relevance to this care profile matters, not count.

Composite score = sum of 4 dimension scores. Maximum = 10.0.

STEP 4 — RANK AND RETURN TOP 3
Before writing the JSON: write out all scored helpers as a list sorted by composite score, highest first. Assign rank 1 to the first helper on that list, rank 2 to the second, rank 3 to the third. If a helper with composite 6.0 appears below helpers with composite 5.6 in your draft, correct it — a higher composite always outranks a lower composite regardless of the order you scored them.
Tie-breaking rule: if two helpers have equal composite scores, the one with the LOWER hourly_rate ranks first. If rates are also equal, the one with MORE years_of_experience ranks first.
Return only the top 3.

Return ONLY a valid JSON object in this exact format — no text before it, no text after it:
{
  "no_match": false,
  "step_reasoning": {
    "budget_gate": "one sentence — threshold calculated and helpers excluded, or skipped if null",
    "deal_breaker_summary": "one sentence — which helpers were excluded and why, with deal-breaker type",
    "schedule_viability": "one sentence — whether non-standard schedule detected; if no-match triggered, which soft deal-breakers suspended and helpers restored",
    "scoring_notes": "one sentence — key factor that separated the top scorers from the rest"
  },
  "excluded_by_deal_breaker": [
    {
      "helper_id": "...",
      "name": "...",
      "deal_breaker_triggered": "exact deal-breaker text from care profile",
      "deal_breaker_type": "hard or soft",
      "reasoning": "one sentence explaining the inference"
    }
  ],
  "top_3": [
    {
      "rank": 1,
      "helper_id": "...",
      "name": "...",
      "hourly_rate": 0,
      "scores": {
        "schedule_fit": 0.0,
        "care_experience": 0.0,
        "skills_match": 0.0,
        "qualifications_match": 0.0
      },
      "composite_score": 0.0,
      "profile_gaps": [],
      "score_reasoning": "one sentence summarising why this helper scored as they did"
    }
  ]
}

Rules:
- hourly_rate: always populate directly from the helper's input profile. Never return null if the value exists in the input data.
- Do not penalise helpers for missing fields that are irrelevant to this specific care profile.
- Only note a field in profile_gaps if it was missing, unresolvable, and caused a dimension to score 0.
- Only exclude a helper on a deal-breaker if it is clearly confirmed or strongly inferable — not on weak or ambiguous signals."""


def run_agent_2(care_profile: dict, family_id: str) -> dict:
    """Run Agent 2 on a single care profile + helpers pool. Returns top 3 scored helpers."""
    user_message = f"""Care profile:
{json.dumps(care_profile, indent=2)}

Helper profiles:
{helpers_json_str}"""

    message = client.messages.create(
        model="claude-sonnet-4-6",
        max_tokens=8192,
        messages=[
            {"role": "user", "content": f"{AGENT_2_PROMPT}\n\n{user_message}"}
        ]
    )
    raw = message.content[0].text.strip()
    import re
    json_block = re.search(r'```json\s*([\s\S]*?)\s*```', raw)
    if json_block:
        raw = json_block.group(1)
    elif raw.startswith("```"):
        raw = raw.split("```")[1]
        if raw.startswith("json"):
            raw = raw[4:]
    try:
        parsed = json.loads(raw)
    except json.JSONDecodeError:
        parsed = {"error": "failed to parse JSON", "raw": raw}
    return {"family_id": family_id, "matching_result": parsed}


print("Agent 2 function defined.")

Agent 2 function defined.


In [7]:
# Run Agent 2 on all families using Agent 1 outputs
agent_2_outputs = []

for a1_output in agent_1_outputs:
    family_id = a1_output["family_id"]
    care_profile = a1_output["care_profile"]
    
    if "error" in care_profile:
        print(f"{family_id}: SKIPPED — Agent 1 output has error")
        continue
    
    print(f"Running Agent 2 on {family_id}...", end=" ")
    result = run_agent_2(care_profile, family_id)
    agent_2_outputs.append(result)
    
    top_3 = result["matching_result"].get("top_3", [])
    top_ids = [h.get("helper_id") for h in top_3]
    print(f"top 3: {top_ids}")
    time.sleep(1)  # avoid rate limiting

# Save outputs
with open(OUTPUT_DIR / "agent_2_outputs.json", "w") as f:
    json.dump(agent_2_outputs, f, indent=2)

print(f"\nAgent 2 complete. Outputs saved to {OUTPUT_DIR / 'agent_2_outputs.json'}")

Running Agent 2 on F001... top 3: ['H001', 'H004', 'H007']
Running Agent 2 on F002... top 3: ['H003', 'H002', 'H009']
Running Agent 2 on F003... top 3: ['H004', 'H002', 'H004']
Running Agent 2 on F004... top 3: ['H002', 'H009', 'H007']
Running Agent 2 on F005... top 3: ['H004', 'H001', 'H005']
Running Agent 2 on F006... top 3: ['H007', 'H004', 'H002']
Running Agent 2 on F007... top 3: ['H003', 'H005', 'H007']
Running Agent 2 on F008... top 3: ['H004', 'H001', 'H007']
Running Agent 2 on F009... top 3: ['H007', 'H010', 'H004']
Running Agent 2 on F010... top 3: ['H002', 'H007', 'H013']
Running Agent 2 on F011... top 3: ['H004', 'H005', 'H001']
Running Agent 2 on F012... top 3: ['H001', 'H006', 'H009']
Running Agent 2 on F013... top 3: ['H003']
Running Agent 2 on F014... top 3: ['H001', 'H005', 'H009']
Running Agent 2 on F015... top 3: ['H019', 'H016', 'H020']
Running Agent 2 on F016... top 3: ['H004', 'H007', 'H001']
Running Agent 2 on F017... top 3: ['H014']

Agent 2 complete. Outputs sa

In [10]:
# Spot check — inspect one Agent 2 result
INSPECT_FAMILY = "F013"  # change this to inspect a different family

result = next(r for r in agent_2_outputs if r["family_id"] == INSPECT_FAMILY)
print(json.dumps(result, indent=2))

{
  "family_id": "F013",
  "matching_result": {
    "no_match": false,
    "step_reasoning": {
      "budget_gate": "Upper bound $33 \u00d7 1.20 = $39.60 threshold; H012 Robert Jensen ($40/hr) excluded as the only helper exceeding the threshold.",
      "deal_breaker_summary": "H001 (age 52), H004 (age 55), H007 (age 58), H018 (age 55), H020 (age 65) excluded on hard deal-breaker 2 (age outside 25\u201350); H002, H005, H006, H008, H009, H010, H011, H013, H014, H015, H016, H017, H019 excluded on hard deal-breaker 1 (no demonstrated experience with younger adults with progressive physical disabilities); H003 James Okonkwo is the sole survivor, with progressive neurological care experience (Parkinson's, stroke rehab) and appropriate age (45) satisfying both hard deal-breakers.",
      "schedule_viability": "Required schedule is standard weekday daytime (8am\u20132pm); no non-standard hours detected, so schedule viability override was not triggered and no soft deal-breakers were suspended.

## 5. Agent 3 — Match Quality & Trust Agent
**Model:** Claude Sonnet 4.6  
**Prompt version:** v1 (locked)  
**Job:** Write family-facing trust-building explanations for the top 3 helpers

In [15]:
AGENT_3_PROMPT = """You are a senior care coordinator writing match recommendations for a family who is looking for care help.

You will be given:
1. The family's care profile (what they need, who they are, what matters to them)
2. The top helpers selected for this family, in ranked order (1, 2, or 3 helpers)

Your job is to write a short, trust-building explanation for each helper — the explanation the family will read when deciding who to reach out to.

Rules:
- Write in warm, plain English — not clinical, not corporate
- Every claim you make about a helper must be traceable to a specific fact in their profile. Do not invent or embellish.
- Mirror the family's own language where possible. If they described a need in a particular way, reflect that framing back.
- Acknowledge that this is not a routine decision. The family is trusting someone with a person they love.
- Do not show scores, rankings as numbers, or any system language. The family should feel they are reading a thoughtful human recommendation, not an algorithm output.
- Keep each explanation to 3–4 sentences. Specific and concise beats long and generic.
- Write one recommendation per helper provided. If only 1 or 2 helpers are given, return 1 or 2 recommendations — do not invent helpers to fill 3 slots.

Return ONLY a JSON object in this format:
{
  "recommendations": [
    {
      "rank": 1,
      "helper_id": "...",
      "name": "...",
      "hourly_rate": 0,
      "explanation": "3–4 sentence trust-building explanation grounded in specific profile facts and the family's own language"
    }
  ]
}

The recommendations array contains one entry per helper provided."""


def run_agent_3(care_profile: dict, top_3_helpers: list, family_id: str) -> dict:
    """Run Agent 3 on a care profile + top helpers. Returns family-facing explanations."""
    helper_lookup = {h["helper_id"]: h for h in helpers}
    enriched_top_3 = []
    for match in top_3_helpers:
        hid = match.get("helper_id")
        if hid in helper_lookup:
            enriched_top_3.append({
                "rank": match["rank"],
                "score": match.get("composite_score"),
                "profile": helper_lookup[hid]
            })

    user_message = f"""Family care profile:
{json.dumps(care_profile, indent=2)}

Top helpers (ranked):
{json.dumps(enriched_top_3, indent=2)}"""

    message = client.messages.create(
        model="claude-sonnet-4-6",
        max_tokens=2048,
        messages=[
            {"role": "user", "content": f"{AGENT_3_PROMPT}\n\n{user_message}"}
        ]
    )
    raw = message.content[0].text.strip()
    if raw.startswith("```"):
        raw = raw.split("```")[1]
        if raw.startswith("json"):
            raw = raw[4:]
    try:
        parsed = json.loads(raw)
    except json.JSONDecodeError:
        parsed = {"error": "failed to parse JSON", "raw": raw}
    return {"family_id": family_id, "recommendations": parsed}


print("Agent 3 function defined.")

Agent 3 function defined.


In [16]:
# Run Agent 3 on all families using Agent 1 + Agent 2 outputs
agent_3_outputs = []

# Build lookup maps
a1_lookup = {r["family_id"]: r["care_profile"] for r in agent_1_outputs}
a2_lookup = {r["family_id"]: r["matching_result"] for r in agent_2_outputs}

for family_id, matching_result in a2_lookup.items():
    care_profile = a1_lookup.get(family_id)
    top_3 = matching_result.get("top_3", [])

    if not top_3:
        print(f"{family_id}: SKIPPED — no top_3 from Agent 2")
        continue
    if not care_profile or "error" in care_profile:
        print(f"{family_id}: SKIPPED — Agent 1 output has error")
        continue

    print(f"Running Agent 3 on {family_id}...", end=" ")
    result = run_agent_3(care_profile, top_3, family_id)
    agent_3_outputs.append(result)
    print("done")
    time.sleep(1)

# Save outputs
with open(OUTPUT_DIR / "agent_3_outputs.json", "w") as f:
    json.dump(agent_3_outputs, f, indent=2)

print(f"\nAgent 3 complete. Outputs saved to {OUTPUT_DIR / 'agent_3_outputs.json'}")

Running Agent 3 on F001... done
Running Agent 3 on F002... done
Running Agent 3 on F003... done
Running Agent 3 on F004... done
Running Agent 3 on F005... done
Running Agent 3 on F006... done
Running Agent 3 on F007... done
Running Agent 3 on F008... done
Running Agent 3 on F009... done
Running Agent 3 on F010... done
Running Agent 3 on F011... done
Running Agent 3 on F012... done
Running Agent 3 on F013... done
Running Agent 3 on F014... done
Running Agent 3 on F015... done
Running Agent 3 on F016... done
Running Agent 3 on F017... done

Agent 3 complete. Outputs saved to outputs/pipeline/agent_3_outputs.json


In [17]:
# Spot check — read Agent 3 explanations for one family
INSPECT_FAMILY = "F017"  # change this to inspect a different family

result = next(r for r in agent_3_outputs if r["family_id"] == INSPECT_FAMILY)
recs = result["recommendations"].get("recommendations", [])
for rec in recs:
    print(f"Rank {rec['rank']} — {rec['name']} (${rec['hourly_rate']}/hr)")
    print(rec["explanation"])
    print()

Rank 1 — Diana Patel ($19/hr)
Diana's connection to this kind of care runs deep — she spent nine years as the primary caregiver for her own mother through all stages of Alzheimer's, which means she understands dementia not just as a condition to manage, but as something she has lived alongside with patience and love. She describes herself as calm and deeply committed, and her background reflects exactly the non-clinical, companion-focused manner your family is looking for. She speaks Gujarati and Hindi alongside English, which may feel like a meaningful comfort for your grandmother day to day. Her weekday morning availability — 7:30am to 12:30pm, Monday through Friday — aligns well with what you've described, though we'd recommend confirming the exact hours and days you need before you connect.



## 6. Pipeline Summary
Quick check of what ran successfully vs. what had errors.

In [18]:
import json                                                                                           
from pathlib import Path                                                                              
                                                                                                        
OUTPUT_DIR = Path("outputs/pipeline")
                                                                                                        
with open(OUTPUT_DIR / "agent_1_outputs.json") as f:
    agent_1_outputs = json.load(f)                  
                                          
with open(OUTPUT_DIR / "agent_2_outputs.json") as f:
      agent_2_outputs = json.load(f)                                                                    
                                    
with open(OUTPUT_DIR / "agent_3_outputs.json") as f:                                                  
      agent_3_outputs = json.load(f)                  
                                          
print(f"Loaded: {len(agent_1_outputs)} A1, {len(agent_2_outputs)} A2, {len(agent_3_outputs)} A3")

Loaded: 17 A1, 17 A2, 17 A3


In [19]:
print("=== PIPELINE SUMMARY ===")
print(f"Families in dataset:      {len(families)}")
print(f"Agent 1 outputs:          {len(agent_1_outputs)}")
print(f"Agent 2 outputs:          {len(agent_2_outputs)}")
print(f"Agent 3 outputs:          {len(agent_3_outputs)}")

a1_errors = [r for r in agent_1_outputs if "error" in r.get("care_profile", {})]
a2_errors = [r for r in agent_2_outputs if "error" in r.get("matching_result", {})]
a3_errors = [r for r in agent_3_outputs if "error" in r.get("recommendations", {})]

print(f"\nAgent 1 parse errors:     {len(a1_errors)}")
print(f"Agent 2 parse errors:     {len(a2_errors)}")
print(f"Agent 3 parse errors:     {len(a3_errors)}")

if a1_errors:
    print(f"  Agent 1 failed families: {[r['family_id'] for r in a1_errors]}")
if a2_errors:
    print(f"  Agent 2 failed families: {[r['family_id'] for r in a2_errors]}")
if a3_errors:
    print(f"  Agent 3 failed families: {[r['family_id'] for r in a3_errors]}")

print("\nAll outputs saved to evals/outputs/pipeline/")
print("Next: run 02_eval_analysis.ipynb")

=== PIPELINE SUMMARY ===
Families in dataset:      17
Agent 1 outputs:          17
Agent 2 outputs:          17
Agent 3 outputs:          17

Agent 1 parse errors:     0
Agent 2 parse errors:     0
Agent 3 parse errors:     0

All outputs saved to evals/outputs/pipeline/
Next: run 02_eval_analysis.ipynb
